# 00 — Environment Setup Verification

**Phase:** Phase 1 — Environment Setup + Data Preparation

**Purpose:** verify that the GPU/CUDA/library environment required for the QLoRA fine-tuning
phases (3+) is actually present and functional, and confirm that the target model's
config and processor load correctly from the Hugging Face Hub.

**IMPORTANT — Colab vs. local distinction:**

This notebook's GPU/CUDA checks are **authoritative only when this notebook is executed on
Google Colab with a GPU runtime**. Per `AGENTS.md` / `CLAUDE.md` §12–13, GPU-dependent
behavior (CUDA availability, device name, VRAM) must be validated on the actual target
execution environment (Colab), not inferred from a local machine.

If this notebook happens to be opened and executed on a local, CPU-only machine (as it was
for the initial authoring/validation pass recorded in this file), the results below are
useful only for **structural validation** — i.e. confirming the notebook runs top-to-bottom
without raising an exception and that the non-GPU library/config/processor checks pass. A
local run reporting `cuda available: False` is an **expected, honest, non-failing result**
on a machine with no CUDA device — it is NOT evidence that the environment is broken, and it
does NOT substitute for real Colab GPU validation. Do not treat a local run of this notebook
as satisfying any GPU-dependent acceptance criterion in `docs/EXPERIMENT_SPEC.md` or
`docs/IMPLEMENTATION_PLAN.md`.

In [1]:
# Detect whether this notebook is running on Google Colab.
# This gates the Colab-only setup cell below and lets every later cell report
# results honestly rather than silently assuming one environment or the other.
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Google Colab: {IN_COLAB}")
print(f"'google.colab' in sys.modules: {'google.colab' in sys.modules}")

Running on Google Colab: True
'google.colab' in sys.modules: True


## Colab-only repository setup

Colab starts each session with a fresh VM, so the repository and this project's package
(`vlm_lab`) are not present until explicitly obtained and installed. This step is skipped
entirely when `IN_COLAB` is `False` (e.g. on this local machine, where the package is
already installed editable in `.venv`), so re-running this notebook locally never tries to
`git clone` or reinstall anything.

Two ways to get the repository onto the Colab VM:

1. **`git clone`** (default below) — `REPO_URL` is set to this repository's GitHub remote,
   and `GIT_REF` pins the branch to check out (currently the pull-request branch this Phase 1
   work lives on, since it has not merged to the default branch yet — see the `TODO` in the
   next cell). The cell is idempotent: if `repo/` already exists on disk (e.g. after a kernel
   restart on the same Colab VM, which does not wipe the filesystem), it fetches and checks
   out `GIT_REF` again and pulls instead of re-cloning, so **Restart & Run All** works
   correctly on a second pass.
2. **Upload / Google Drive** — alternatively, upload the repo as a zip via the Colab file
   browser, or mount Google Drive (`google.colab.drive.mount`) if the repo already lives
   there, then `%cd` into it. Drive mounting is **not** a hard requirement of this notebook —
   it is just one option for getting the code onto the VM.

In [ ]:
REPO_URL = "https://github.com/Mr-Kondo/finetuning_vlm.git"

# PR #1 merged (2026-08-11, commit 72dd7b7) -- main now has this work directly,
# so a plain clone of the default branch is sufficient.
GIT_REF = ""  # "" = default branch

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess
    import sys

    if not REPO_URL:
        raise RuntimeError(
            "IN_COLAB is True but REPO_URL is not set. Set REPO_URL above before "
            "running this notebook on Colab — later cells (model config / processor "
            "loading) depend on `vlm_lab` being installed, so continuing without a "
            "repository would leave the notebook in a silently broken state."
        )

    repo_dir = "repo"

    if os.path.isdir(repo_dir):
        # A kernel restart on the same Colab VM leaves /content (and this repo
        # checkout) on disk even though Python state resets. A plain `git clone`
        # would then fail because the destination already exists, breaking the
        # required Restart-and-Run-All workflow. Update in place instead.
        print(f"'{repo_dir}' already exists — updating instead of re-cloning "
              "(idempotent across kernel restarts).")
        fetch = subprocess.run(
            ["git", "-C", repo_dir, "fetch", "origin"], capture_output=True, text=True
        )
        if fetch.returncode != 0:
            raise RuntimeError(
                f"git fetch failed (exit code {fetch.returncode}):\n{fetch.stderr}"
            )
        if GIT_REF:
            checkout = subprocess.run(
                ["git", "-C", repo_dir, "checkout", GIT_REF], capture_output=True, text=True
            )
            if checkout.returncode != 0:
                raise RuntimeError(
                    f"git checkout {GIT_REF!r} failed (exit code {checkout.returncode}):\n"
                    f"{checkout.stderr}"
                )
        pull = subprocess.run(
            ["git", "-C", repo_dir, "pull"], capture_output=True, text=True
        )
        if pull.returncode != 0:
            raise RuntimeError(
                f"git pull failed (exit code {pull.returncode}):\n{pull.stderr}"
            )
    else:
        clone_cmd = ["git", "clone"]
        if GIT_REF:
            clone_cmd += ["--branch", GIT_REF]
        clone_cmd += [REPO_URL, repo_dir]
        clone = subprocess.run(clone_cmd, capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit code {clone.returncode}):\n{clone.stderr}"
            )

    os.chdir(repo_dir)

    # NOTE: do not add ad-hoc `pip install --upgrade ...` calls for other
    # packages here (e.g. pyarrow) "to fix an error you saw" -- this project's
    # dependencies are exact-pinned in pyproject.toml (EXPERIMENT_SPEC.md §7).
    # Upgrading an unpinned package here silently breaks that pin. If a real
    # incompatibility is found, fix the pin in pyproject.toml itself so it's
    # tracked, reviewed, and consistent for every future run -- not papered
    # over inside this cell.
    #
    # Use `sys.executable -m pip`, not a bare `pip` command: Colab can have
    # more than one Python/pip on PATH, and a bare `pip` invocation can
    # install into a different interpreter's site-packages than the one
    # actually running this notebook.
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
        capture_output=True, text=True,
    )
    # Always show pip's actual resolution log, not just on failure -- this is
    # the only way to see whether pip genuinely reinstalled a pinned package
    # to the exact pinned version, versus deciding an already-installed
    # version was "close enough" for some transitive reason.
    print("--- pip install output ---")
    print(install.stdout)
    if install.stderr:
        print(install.stderr)
    print("--- end pip install output ---")
    if install.returncode != 0:
        raise RuntimeError(f"pip install failed (exit code {install.returncode}).")

    # An editable install performed *after* this interpreter already started
    # writes a new `.pth` file into site-packages, but Python's `site` module
    # only processes `.pth` files at interpreter startup -- so, even with the
    # correct interpreter above, `import vlm_lab` still fails right after
    # install with a plain `ModuleNotFoundError` (confirmed by direct local
    # reproduction: `sys.executable -m pip install -e .` followed immediately
    # by `importlib.import_module` fails, and `importlib.invalidate_caches()`
    # alone does not fix it either). `site.addsitedir()` re-processes each
    # site-packages directory, including the newly written `.pth` file, which
    # does fix it (also confirmed by direct local reproduction).
    for site_dir in site.getsitepackages():
        site.addsitedir(site_dir)
    importlib.invalidate_caches()

    try:
        importlib.import_module("vlm_lab")
    except ImportError as exc:
        raise RuntimeError(
            "Repository was cloned and `pip install -e .[dev]` reported success, "
            f"but `import vlm_lab` still failed: {exc}"
        ) from exc

    print("Repository cloned, package installed, and `import vlm_lab` succeeded.")
else:
    print("Not running on Colab — skipping repo clone / install (package already installed locally).")

## Python and core library versions

Later phases (QLoRA training, evaluation) depend on specific behavior of `torch`,
`transformers`, `datasets`, and `Pillow`. Recording the actually-installed versions here
(rather than assuming versions from `pyproject.toml`'s lower bounds) is part of the
reproducibility record required by `CLAUDE.md` §11.

In [ ]:
import platform

import torch
import transformers
import datasets
import PIL

print(f"Python version: {platform.python_version()}")
print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"Pillow (PIL) version: {PIL.__version__}")

## CUDA availability

4-bit QLoRA training requires a CUDA GPU. This cell must never raise, whether or not a
CUDA device is present, so that it behaves correctly both on this local CPU-only machine and
on a Colab GPU runtime.

In [4]:
cuda_available = torch.cuda.is_available()
print(f"torch.cuda.is_available(): {cuda_available}")

if cuda_available:
    print(f"torch.version.cuda: {torch.version.cuda}")
    print(f"CUDA device count: {torch.cuda.device_count()}")
else:
    print("No CUDA device available: torch.version.cuda and device count are not applicable.")

torch.cuda.is_available(): True
torch.version.cuda: 13.0
CUDA device count: 1


## GPU identity and memory (if available)

Knowing the actual assigned GPU (name, total VRAM) is required later to reason about
whether 4-bit QLoRA fine-tuning of a 4B-parameter VLM will fit in memory (see
`docs/DECISIONS.md` ADR-014, the production-shape VRAM go/no-go gate). This cell reports
a clear message instead of crashing when no CUDA device is present.

In [5]:
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_memory_gib = props.total_memory / (1024 ** 3)
    print(f"GPU name: {device_name}")
    print(f"Total GPU memory: {total_memory_gib:.2f} GiB")
else:
    device_name = None
    total_memory_gib = None
    print("No CUDA device available: skipping GPU name / memory query.")

GPU name: Tesla T4
Total GPU memory: 14.56 GiB


## Target model config

Loading only the model **config** (a small JSON file from the Hub) confirms that the model
identifier is correct and reachable, and that the installed `transformers` version
recognizes its architecture — without downloading the multi-gigabyte model weights, which
belongs to Phase 3+. `AutoConfig` (rather than a `Qwen3VL*`-specific class) is used for
forward-compatible, standard loading.

In [6]:
from transformers import AutoConfig

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

model_config = AutoConfig.from_pretrained(MODEL_ID)

print(f"Config class: {type(model_config).__name__}")
print(f"model_type: {getattr(model_config, 'model_type', None)}")
print(f"architectures: {getattr(model_config, 'architectures', None)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

Config class: Qwen3VLConfig
model_type: qwen3_vl
architectures: ['Qwen3VLForConditionalGeneration']


## Target model processor

The processor (tokenizer + image processor) is also lightweight relative to the model
weights and is required for every later phase that builds prompts or preprocesses images.
Loading it here confirms the processor files are present on the Hub and compatible with the
installed `transformers` version, using the standard `AutoProcessor` entry point.

In [ ]:
# NOTE: if this raises a PIL/torchvision-related ImportError (e.g. involving
# `PIL._typing._Ink`), do not "fix" it by reinstalling torch/torchvision/Pillow
# ad hoc inside this cell (including via an AI-suggested auto-fix) -- that
# breaks pyproject.toml's exact version pins (EXPERIMENT_SPEC.md §7) and, on
# a persistent Colab VM, leaves packages downgraded on disk even after a
# kernel "Restart session" (a kernel restart does not undo `pip install`
# effects). If you hit this, do a full Runtime -> Disconnect and delete
# runtime (not just Restart session) to get a clean disk, re-run this
# notebook's install cell so pyproject.toml's exact pins are installed
# fresh, and if it still fails, report the real traceback so the pin
# itself can be fixed and reviewed.
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Processor class: {type(processor).__name__}")
print(f"Processor summary: {processor}")

## Environment summary

A single collected record of the reproducibility-relevant facts gathered above. This is a
plain printed dict for at-a-glance review while running this notebook — not a persistence or
logging framework (out of scope for Phase 1; YAGNI).

In [ ]:
environment_summary = {
    "in_colab": IN_COLAB,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "datasets_version": datasets.__version__,
    "pillow_version": PIL.__version__,
    "cuda_available": cuda_available,
    "cuda_version": torch.version.cuda if cuda_available else None,
    "gpu_device_name": device_name,
    "gpu_total_memory_gib": total_memory_gib,
    "model_id": MODEL_ID,
    "model_config_class": type(model_config).__name__,
    "processor_class": type(processor).__name__,
}

for key, value in environment_summary.items():
    print(f"{key}: {value}")